# Thermal Real-ESRGAN — Colab training (160×120 → 640×480)

Fine-tunes **SRVGGNetCompact 64/16** (the architecture of `realesr-general-x4v3`) on thermal
imagery with sensor-specific degradation, then converts it to ncnn fp16 for the Android app.

**Before you start, put this on your Drive:**

```
MyDrive/thermal_sr/
├── raw/                 HR thermal images (FLIR ADAS v2 / KAIST / PBVS / anything ≥480px)
│   ├── flir_adas_v2/
│   └── kaist/
├── val_mag160/          ~20 real 160×120 captures from your camera (optional but recommended)
└── captures/            wall_300.npy — (N,H,W) static frames for FPN measurement (optional)
```

Runtime → Change runtime type → **T4 GPU** (or better).

Everything (checkpoints, logs, exports) is written to Drive, and every training cell uses
`--auto_resume`, so a disconnect costs you at most the iterations since the last checkpoint —
just re-run the cell.


# Thermal Real-ESRGAN — Colab training (160×120 → 640×480)

Fine-tunes a 1-channel **SRVGGNetCompact 64/10** (from `realesr-general-x4v3`) on thermal imagery
seen the way the app sees it, then converts it to an ncnn fp16 zip the app imports directly.

Training data downloads itself from public Hugging Face mirrors — no dataset account, and
**Drive is optional** (`USE_DRIVE` in the next cell).

- `USE_DRIVE = False` (default): everything on the runtime disk. Nothing to authorise, but
  a disconnect wipes it — use the *save progress* cell to pull checkpoints down as you go.
- `USE_DRIVE = True`: checkpoints survive disconnects and training resumes where it left off —
  **recommended for the full run** (~6 h on an A100).

Optional inputs, if you have them: real 160×120 captures in `<root>/val_mag160/` for the
hallucination check (the file panel's `thermal_sr/val_mag160/`, or `MyDrive/thermal_sr/…` with
Drive; `scripts/mgt_to_png.py` makes them from the app's .mgt files; upload after the next cell), and a static-scene `(N,H,W)` .npy in `<root>/captures/` to measure your
sensor's fixed-pattern noise.

Runtime → Change runtime type → **A100** if you have it (a T4 works, several times slower).


In [ ]:
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, '
                     'then re-run. Training cannot fall back to CPU: Real-ESRGAN calls '
                     '.cuda() directly (USMSharp, the degradation pipeline).')
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name}, {props.total_memory / 1024**3:.1f} GB')
if props.total_memory / 1024**3 < 14:
    print('WARNING: under ~14 GB — drop batch_size to 6 and queue_size to 80 in the '
          'config cell, or stage 2 will run out of memory.')

# ---------------------------------------------------------------------------
USE_DRIVE = False   # True mounts Drive and keeps everything there across sessions
# ---------------------------------------------------------------------------

import pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = pathlib.Path('/content/drive/MyDrive/thermal_sr')
else:
    ROOT = pathlib.Path('/content/thermal_sr')
    print('\nNOT using Drive. Everything lives on the runtime disk, which Colab wipes when',
          '\nthe session ends or disconnects — checkpoints included. Run the "save progress"',
          '\ncell now and then to pull the latest checkpoint down to your machine.')

for sub in ('raw', 'val_mag160', 'captures', 'experiments', 'export', 'cache'):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
print('\nroot:', ROOT)
print('raw sources:', [p.name for p in (ROOT / 'raw').iterdir() if p.is_dir()])


## 2. Install Real-ESRGAN + this project's pipeline

Two things need patching before anything installs, both the same Python 3.13 issue:
`basicsr`'s and Real-ESRGAN's `setup.py` read their version with `exec()` and then
`locals()['__version__']`, which PEP 667 broke — you get `KeyError: '__version__'`.
So we patch basicsr's sdist before installing it, and skip Real-ESRGAN's `setup.py`
entirely by running the trainer from the repo directory instead of installing it.


In [ ]:
%cd /content
!rm -rf Real-ESRGAN MAG160_ThermalCam
!git clone -q https://github.com/xinntao/Real-ESRGAN.git
!git clone -q https://github.com/delphicchen/MAG160_ThermalCam.git
%cd /content/Real-ESRGAN
!cp -r /content/MAG160_ThermalCam/sr_train/options /content/MAG160_ThermalCam/sr_train/thermal_arch /content/MAG160_ThermalCam/sr_train/scripts .

# runtime deps (torch/torchvision/opencv/scipy ship with Colab)
!pip -q install addict yapf lmdb future tqdm onnx onnxsim ncnn pnnx tb-nightly

# --- basicsr ---------------------------------------------------------------
# basicsr's setup.py reads its version with exec() + locals()['__version__'], which
# PEP 667 broke on Python 3.13. pip cannot even *download* it with --no-binary,
# because that runs setup.py egg_info first — so fetch the sdist straight from PyPI,
# patch it, then install from the patched directory.
import json, pathlib, shutil, subprocess, sys, tarfile, urllib.request

work = pathlib.Path('/tmp/bs')
shutil.rmtree(work, ignore_errors=True)
work.mkdir(parents=True)

meta = json.load(urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json'))
url = next(u['url'] for u in meta['urls'] if u['packagetype'] == 'sdist')
tgz = work / 'basicsr.tar.gz'
urllib.request.urlretrieve(url, tgz)
with tarfile.open(tgz) as t:
    t.extractall(work, filter='data')
pkg = next(p for p in work.iterdir() if p.is_dir())

sp = pkg / 'setup.py'
src = sp.read_text()
before = src
src = src.replace("exec(compile(f.read(), version_file, 'exec'))",
                  "_ns = {}\n        exec(compile(f.read(), version_file, 'exec'), _ns)")
src = src.replace("return locals()['__version__']", "return _ns['__version__']")
assert src != before, 'setup.py did not match the expected pattern — inspect it manually'
sp.write_text(src)
print('patched', sp)

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(pkg), '--no-deps'],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
r.check_returncode()

# --- torchvision >= 0.17 moved functional_tensor; basicsr still imports the old path ---
# Find the package WITHOUT importing it: importing basicsr executes the very line that
# is broken, so `import basicsr` cannot come before the patch.
import importlib.util
spec = importlib.util.find_spec('basicsr')
assert spec and spec.origin, 'basicsr is not importable — the install above failed'
pkg_dir = pathlib.Path(spec.origin).parent
hit = []
for f in pkg_dir.rglob('*.py'):          # degradations.py is the known one; catch any other
    t = f.read_text()
    if 'functional_tensor' in t:
        f.write_text(t.replace('torchvision.transforms.functional_tensor',
                               'torchvision.transforms.functional'))
        hit.append(f.name)
print('patched:', hit)

# --- realesrgan/version.py is generated by setup.py, which we deliberately skip ---
# realesrgan/__init__.py does `from .version import *`, so the package needs one.
ver = pathlib.Path('realesrgan/version.py')
if not ver.exists():
    v = pathlib.Path('VERSION').read_text().strip() if pathlib.Path('VERSION').exists() else '0.3.0'
    ver.write_text(f"__version__ = '{v}'\n__gitsha__ = 'unknown'\n"
                   f"version_info = tuple(int(x) for x in '{v}'.split('.')[:3])\n")
    print('wrote', ver, v)

# --- register our degradation models with BasicSR's registry ---
# `python realesrgan/train.py` puts realesrgan/ first on sys.path, and that directory
# has its own `archs` package — a top-level `archs` of ours would be shadowed by it
# (ModuleNotFoundError: No module named 'archs.thermal_degradation'). Hence thermal_arch,
# plus an explicit repo-root insert so the import does not depend on PYTHONPATH.
p = pathlib.Path('realesrgan/train.py')
s = p.read_text()
if 'thermal_arch' not in s:
    s = s.replace("import os.path as osp",
                  "import os.path as osp\n"
                  "import sys as _sys\n"
                  "_sys.path.insert(0, osp.dirname(osp.dirname(osp.abspath(__file__))))\n"
                  "import thermal_arch.thermal_degradation  # noqa: F401", 1)
    p.write_text(s)
print(pathlib.Path('realesrgan/train.py').read_text().splitlines()[:8])

# --- verify before moving on ----------------------------------------------
import basicsr
from basicsr.data.degradations import circular_lowpass_kernel  # noqa: F401
from basicsr.utils.registry import MODEL_REGISTRY
import thermal_arch.thermal_degradation  # noqa: F401
import torch
print('basicsr', basicsr.__version__, '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())
print('registered:', [k for k in MODEL_REGISTRY._obj_map if 'Thermal' in k])


## 3. Get thermal imagery — no account needed

FLIR's own ADAS download and the PBVS challenge data both need registration. These
Hugging Face mirrors do not — each was checked to be public and ungated:

| source | images | size | note |
|---|---|---|---|
| `jsonhash/FLIR_aligned` | ~5.1k thermal @ **640×512** | 1.4 GB zip | FLIR ADAS aligned pairs; we take only `*_PreviewData.jpeg` |
| `vision-cidis/CIDIS-dataset` (GitHub) | 700 train + 200 val @ **640×448** | ~0.8 GB | PBVS TISR 2024/25 benchmark; grey, milder AGC than FLIR's previews. No licence file — its README asks for a citation (Rivadeneira et al., 2024) |
| `LibreYOLO/flir-camera-objects` | 13.6k @ 640×640 | ~1 GB | same FLIR ADAS via Roboflow, **stretched** to square — off by default |
| `Kiuyha/hit-uav-thermal-human-detection` | 4.9k @ 640×640 | ~300 MB | drone thermal, also a Roboflow export (`.rf.`), stretched — off by default |

FLIR aligned + CIDIS train is the default. CIDIS val stays out of training and becomes the
section 8 check set when you have no real captures. Raw images land in `/content/raw` (local disk — Drive
is far too slow for thousands of small files) and the zip is cached on Drive so a
reconnect does not re-download it. Anything you put in `Drive/thermal_sr/raw/` is picked
up as an extra source.


In [ ]:
import pathlib, shutil, zipfile
from huggingface_hub import hf_hub_download, snapshot_download

EXTRA_SOURCES = False   # True also pulls the two Roboflow sets (~1.3 GB, stretched 640x640)

raw = pathlib.Path('/content/raw'); raw.mkdir(parents=True, exist_ok=True)
cache = ROOT / 'cache'; cache.mkdir(exist_ok=True)

# --- FLIR aligned: native 640x512 thermal, the best of the three ---
zp = cache / 'flir_aligned.zip'
if not zp.exists():
    got = hf_hub_download('jsonhash/FLIR_aligned', 'aligned.zip',
                          repo_type='dataset', local_dir='/content/hf')
    shutil.copy(got, zp)          # keep on Drive: survives a disconnect
    print('cached', zp)

dst = raw / 'flir_aligned'; dst.mkdir(exist_ok=True)
if not any(dst.iterdir()):
    with zipfile.ZipFile(zp) as z:
        members = [n for n in z.namelist() if n.endswith('_PreviewData.jpeg')]
        print(f'extracting {len(members)} thermal frames…')
        for n in members:
            with z.open(n) as src, open(dst / pathlib.Path(n).name, 'wb') as out:
                shutil.copyfileobj(src, out)
print('flir_aligned:', len(list(dst.glob('*.jpeg'))), 'images')

if EXTRA_SOURCES:
    for repo, sub in (('LibreYOLO/flir-camera-objects', 'flir_roboflow'),
                      ('Kiuyha/hit-uav-thermal-human-detection', 'hit_uav')):
        d = raw / sub
        if not d.exists():
            snapshot_download(repo, repo_type='dataset', local_dir=str(d),
                              allow_patterns=['*/images/*.jpg'])
        print(sub, len(list(d.rglob('*.jpg'))), 'images')

# --- CIDIS: PBVS TISR 2024/25 benchmark, public on GitHub, no account -------------------
# Only the thermal train/val folders are checked out (sparse, blobless). train joins the
# crops; val is kept apart — section 8 uses it as the check set if you have no captures.
import subprocess
cidis = pathlib.Path('/content/cidis')
if not (cidis / 'dataset/thermal/train').exists():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--sparse',
                    'https://github.com/vision-cidis/CIDIS-dataset.git', str(cidis)], check=True)
    subprocess.run(['git', '-C', str(cidis), 'sparse-checkout', 'set',
                    'dataset/thermal/train', 'dataset/thermal/val'], check=True)
d = raw / 'cidis'
if not d.exists():
    shutil.copytree(cidis / 'dataset/thermal/train', d)
print('cidis:', len(list(d.glob('*.bmp'))), 'train images;',
      len(list((cidis / 'dataset/thermal/val').glob('*.bmp'))), 'val images kept apart')

# --- anything you dropped on Drive yourself ---
if any((ROOT / 'raw').iterdir()):
    shutil.copytree(ROOT / 'raw', raw / 'user', dirs_exist_ok=True)
    print('user:', len(list((raw / 'user').rglob('*'))), 'entries')

!du -sh /content/raw/*


### Cut them into HR crops

640×512 and 640×640 sources give a single 480px tile at stride 360, so the stride is
shortened to overlap them into several — that is what gets the crop count up.


In [ ]:
!PYTHONPATH=/content/Real-ESRGAN python scripts/prepare_thermal_dataset.py \
    --raw /content/raw \
    --out datasets/thermal_hr \
    --meta datasets/meta_info/thermal_hr.txt \
    --crop 480 --stride 160


In [ ]:
# A run with a handful of images memorises them and teaches you nothing — check before
# spending hours on it.
import pathlib
n = len(pathlib.Path('datasets/meta_info/thermal_hr.txt').read_text().split())
print(f'{n} HR crops')
if n < 200:
    raise SystemExit('Too few crops to train on. Is Drive/thermal_sr/raw/ actually '
                     'populated with HR thermal images of at least 480x480? Sub-folders '
                     'or loose files both work.')
if n < 5000:
    print('WARNING: under ~5k crops the GAN stage will overfit — more sources would help.')


## 4. Check your sensor's noise (optional)

The degradation adds sensor noise in °C for a scene of random temperature span, the way the app
sees it (`sensor_*` in the config cell). The defaults are generic ranges that already cover real
MAG160 frames with margin — run this only to see whether *your* unit is noisier: record a still,
uniform scene with "Save temperature data" on, turn the .mgt into an (N, H, W) .npy of °C, and put
it in `Drive/thermal_sr/captures/`.


In [ ]:
import pathlib
cap = list((ROOT / 'captures').glob('*.npy'))
if cap:
    !PYTHONPATH=/content/Real-ESRGAN python scripts/estimate_fpn_stats.py {cap[0]}
else:
    print('no captures/*.npy — using the default sensor ranges')


## 5. Colab-tuned configs

The repo ymls target a 24 GB card. This cell picks the profile from the GPU it finds:

| GPU | `gt_size` | batch | queue | data workers |
|---|---|---|---|---|
| T4 (16 GB) | 192 | 8 | 120 | 2 |
| A100 (40/80 GB) | 256 | 12 | 180 | up to 8 |

Both run stage 1 for 8k iterations and stage 2 for 40k (the ymls say 10k / 100k). 40k with the
pretrained init is enough to see the real behaviour; push `STAGE2_ITER` further only if the
validation numbers are still improving. On an A100, stage 2 speeds up the most (the
discriminator and VGG loss are the heavy part); stage 1 is mostly limited by data loading.

Keep one GPU per run: `--auto_resume` after switching GPUs picks up the checkpoint, but the
batch size changes under it.


In [ ]:
import yaml, pathlib

# --- sensor view, °C (widen only if section 4 said so) ---------------------
SENSOR = dict(
    sensor_span_c=[0.5, 40.0],     # scene span, log-uniform
    sensor_noise_c=[0.02, 0.20],   # pixel noise
    sensor_noise_log=True,         # log-uniform: the app's temporal denoise leaves most inputs quiet
    sensor_stripe_c=[0.0, 0.10],   # column and row stripes, equally (the app rotates)
    sensor_map_c=[0.0, 0.05],      # static 2-D residual
)
# v4: the whole budget goes to stage 1 (L1 + spectrum + gradient). The light-GAN stage 2
# added texture that was not in the data (v3: 9x the invented peaks), so it is off.
STAGE1_ITER, STAGE2_ITER = 80000, 20000
RUN_STAGE2 = False
# Bump RECIPE when the recipe changes: an experiment left by another recipe (on Drive it
# survives sessions) is archived below instead of silently resumed by --auto_resume.
RECIPE = 'v4-l1-fft1-grad05-noiselog'
# ---------------------------------------------------------------------------

# Batch / crop / workers follow the GPU. The small model barely loads a big card, so on an
# A100 the win comes from larger batches and more data-loader workers, not the GPU alone.
import os, torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
if GPU_GB >= 30:      # A100 40/80 GB, L4 is 22 GB and stays on the T4 profile
    BATCH, GT, QUEUE, WORKERS = 12, 256, 180, min(8, os.cpu_count() or 2)   # the repo ymls
else:                 # T4 16 GB
    BATCH, GT, QUEUE, WORKERS = 8, 192, 120, 2
print(f'{torch.cuda.get_device_name(0)} ({GPU_GB:.0f} GB): batch {BATCH}, gt {GT}, '
      f'queue {QUEUE}, workers {WORKERS}')

def tune(src, dst, total_iter, milestones):
    o = yaml.safe_load(open(src))
    o.update(SENSOR)
    o['gt_size'] = GT
    o['queue_size'] = QUEUE
    o['datasets']['train']['gt_size'] = GT
    o['datasets']['train']['batch_size_per_gpu'] = BATCH
    o['datasets']['train']['num_worker_per_gpu'] = WORKERS
    o['train']['total_iter'] = total_iter
    o['train']['scheduler']['milestones'] = milestones
    o['logger']['save_checkpoint_freq'] = 2000
    yaml.safe_dump(o, open(dst, 'w'), sort_keys=False)
    return dst

tune('options/01_thermal_srvgg_x4_net.yml', 'options/colab_01_net.yml', STAGE1_ITER, [int(STAGE1_ITER*0.6)])
o2 = yaml.safe_load(open('options/02_thermal_srvgg_x4_gan.yml'))
o2['path']['pretrain_network_g'] = 'PLACEHOLDER — the stage 2 cell fills this in'
yaml.safe_dump(o2, open('options/_02_tmp.yml', 'w'), sort_keys=False)
tune('options/_02_tmp.yml', 'options/colab_02_gan.yml', STAGE2_ITER, [int(STAGE2_ITER*0.5), int(STAGE2_ITER*0.8)])

# experiments live under ROOT (Drive when USE_DRIVE, else the runtime disk)
!rm -rf experiments && ln -sfn {ROOT}/experiments experiments
# Download with a real error check: `wget -nc -q` on a 404 leaves no file and says
# nothing, which only surfaces two cells later as a FileNotFoundError.
import pathlib, urllib.request
pm = pathlib.Path('experiments/pretrained_models'); pm.mkdir(parents=True, exist_ok=True)
WEIGHTS = {
    # generator init (3-channel 64/32; truncate_pretrained.py makes it 1-channel 64/10 below)
    'realesr-general-x4v3.pth':
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    # discriminator init — note the tag is v0.2.2.3, not .4
    'RealESRGAN_x4plus_netD.pth':
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.3/RealESRGAN_x4plus_netD.pth',
}
for name, url in WEIGHTS.items():
    f = pm / name
    if not f.exists() or f.stat().st_size < 1_000_000:
        print('downloading', name)
        urllib.request.urlretrieve(url, f)
    assert f.stat().st_size > 1_000_000, f'{name} is missing or truncated'
    print(f'{name}: {f.stat().st_size / 1e6:.1f} MB')

# realesr-general-x4v3 is 3-channel num_conv=32; this recipe runs a 1-channel 64/10. Keep the
# first 10 body convs, move the real output conv onto ours, and fold RGB into one channel
# (the grey net then computes exactly the mean of the RGB net's outputs — a lossless start).
!PYTHONPATH=/content/Real-ESRGAN python scripts/truncate_pretrained.py \
    --src experiments/pretrained_models/realesr-general-x4v3.pth \
    --num-conv 10 --in-ch 1 --out-ch 1 \
    --out experiments/pretrained_models/realesr-general-x4v3-conv10-grey.pth
# stage 2's discriminator sees 1-channel images too: fold its RGB input conv the same way
!PYTHONPATH=/content/Real-ESRGAN python scripts/netd_single_channel.py \
    --src experiments/pretrained_models/RealESRGAN_x4plus_netD.pth \
    --out experiments/pretrained_models/RealESRGAN_x4plus_netD-grey.pth
!ls -la experiments/pretrained_models

# archive runs from another recipe; the marker sits beside the experiment dir because
# BasicSR renames the dir itself when it starts a fresh run
import shutil, time
for stage in ('thermal_srvgg_x4_net', 'thermal_srvgg_x4_gan'):
    d = pathlib.Path('experiments') / stage
    mark = pathlib.Path('experiments') / f'{stage}.recipe'
    if d.exists() and (not mark.exists() or mark.read_text().strip() != RECIPE):
        dst = pathlib.Path('experiments/_archive') / f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}'
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(d), str(dst))
        print(f'archived a run from another recipe: {d} -> {dst}')
    mark.write_text(RECIPE + '\n')
print('recipe:', RECIPE, '| stage 2:', 'on' if RUN_STAGE2 else 'off')


### 5b. Restore a progress zip (only after a lost runtime)

Without Drive, a recycled runtime wipes every checkpoint. If you ran *save progress* before
that, re-run sections 1, 2 and 5 in the new runtime, then upload `thermal_sr_progress.zip` here.

- A restored **stage 2** checkpoint goes straight to sections 8 and 9 (validate, export).
- A restored **stage 1** checkpoint lets you skip stage 1 and run stage 2.

The zip holds generator weights only, not optimizer state, so `--auto_resume` cannot continue a
half-finished stage from it. Re-running that stage starts it over and BasicSR moves the restored
folder aside to `*_archived_*`.


In [ ]:
import os, pathlib, re, shutil, zipfile
os.chdir('/content/Real-ESRGAN')
if not pathlib.Path('experiments').is_symlink():
    raise SystemExit('Run section 5 first — it links experiments/ to the data root, and '
                     'would delete anything restored before it.')

from google.colab import files
for name, data in files.upload().items():
    zp = pathlib.Path('/content') / name
    zp.write_bytes(data)
    with zipfile.ZipFile(zp) as z:
        for n in z.namelist():
            m = re.match(r'(thermal_srvgg_x4_(?:net|gan))_(net_g_.+\.pth)$', pathlib.Path(n).name)
            if not m:
                print('skipped', n)
                continue
            d = pathlib.Path('experiments') / m[1] / 'models'
            d.mkdir(parents=True, exist_ok=True)
            with z.open(n) as s, open(d / m[2], 'wb') as o:
                shutil.copyfileobj(s, o)
            print('restored', d / m[2])


## 6. Stage 1 — L1 + spectrum + gradient, 80k iterations (~5–6 h on an A100)

No adversarial loss: the pixel L1 plus L1s on the image spectrum and on Sobel gradients, which
hold edges without inventing texture. Its last checkpoint is the model. Re-run this cell after a
disconnect; `--auto_resume` picks up the last checkpoint.


In [ ]:
!PYTHONPATH=/content/Real-ESRGAN python realesrgan/train.py -opt options/colab_01_net.yml --auto_resume


In [ ]:
!ls -la experiments/thermal_srvgg_x4_net/models/ || echo 'stage 1 produced no checkpoint'


## 7. Stage 2 — light GAN (optional, off by default)

`RUN_STAGE2 = False` in the config cell skips these two cells. In v3 the light GAN (perceptual 0.3,
GAN 1e-2) made edges crisper but invented 9x the peaks of stage 1 on real frames — kept only
for comparison runs.


In [ ]:
if not globals().get('RUN_STAGE2', False):
    print('Stage 2 is off (RUN_STAGE2 = False in the config cell) — go on to section 8.')
else:
    # Stage 2 starts from whatever stage 1 actually produced — a Colab disconnect can leave
    # the last checkpoint short of the configured total, and a hardcoded iteration number
    # then fails with FileNotFoundError.
    import pathlib, re, yaml

    def iter_of(q):
        """BasicSR also writes net_g_latest.pth, whose name carries no iteration number."""
        m = re.findall(r'\d+', q.stem)
        return int(m[-1]) if m else -1

    ck_dir = pathlib.Path('experiments/thermal_srvgg_x4_net/models')
    cks = sorted(ck_dir.glob('net_g_*.pth'), key=iter_of) if ck_dir.is_dir() else []
    numbered = [c for c in cks if iter_of(c) >= 0]
    pick = numbered[-1] if numbered else (cks[-1] if cks else None)
    if pick is None:
        raise SystemExit('Stage 1 has produced no checkpoint yet — run the previous cell '
                         'to completion first (it writes into ' + str(ck_dir) + ').')

    o = yaml.safe_load(open('options/colab_02_gan.yml'))
    o['path']['pretrain_network_g'] = str(pick)
    yaml.safe_dump(o, open('options/colab_02_gan.yml', 'w'), sort_keys=False)
    print('stage 1 checkpoints:', [c.name for c in cks])
    print('stage 2 starts from:', pick)


In [ ]:
if globals().get('RUN_STAGE2', False):
    get_ipython().system('PYTHONPATH=/content/Real-ESRGAN python realesrgan/train.py '
                         '-opt options/colab_02_gan.yml --auto_resume')
else:
    print('Stage 2 is off — nothing to train here.')


### Save progress (run whenever, especially without Drive)

Zips the newest generator checkpoint and downloads it. Keep the file: re-training costs
hours, converting a checkpoint costs seconds.


In [ ]:
import pathlib, re, shutil

def newest(d):
    q = sorted(pathlib.Path(d).glob('net_g_*.pth'),
               key=lambda f: (int(re.findall(r'\d+', f.stem)[-1]) if re.findall(r'\d+', f.stem) else -1))
    return q[-1] if q else None

out = pathlib.Path('/content/progress'); out.mkdir(exist_ok=True)
for stage in ('thermal_srvgg_x4_net', 'thermal_srvgg_x4_gan'):
    ck = newest(f'experiments/{stage}/models')
    if ck:
        shutil.copy(ck, out / f'{stage}_{ck.name}')
        print('saved', stage, ck.name)
shutil.make_archive('/content/thermal_sr_progress', 'zip', out)
from google.colab import files
files.download('/content/thermal_sr_progress.zip')


## 8. Validate — fake hot spots, energy preservation

Runs `hallucination_check.py` on the newest checkpoint of **both** stages. Prefer the candidate
whose invented peaks stay near zero; between checkpoints of one stage, pick where sharpness has
stopped improving before invented peaks start climbing. The cell rebuilds its own state, so it
also runs after a kernel restart.


In [ ]:
# Restart-safe: a kernel restart wipes ROOT and the working directory set by earlier cells
# (NameError: name 'ROOT' is not defined), so rebuild them from what is on disk.
import os, sys, pathlib, subprocess
if not os.path.isdir('/content/Real-ESRGAN/scripts'):
    raise SystemExit('/content/Real-ESRGAN is gone — the runtime was recycled. Re-run sections '
                     '1, 2 and 5, then restore your progress zip (section 5b).')
sys.path.insert(0, '/content/Real-ESRGAN/scripts')
from colab_state import restore, iter_of
ROOT, _ = restore(None)

# Real MAG160 frames first: scripts/mgt_to_png.py turns the app's .mgt temperature
# captures into 160x120 PNGs for Drive/thermal_sr/val_mag160/.
val = ROOT / 'val_mag160'; val.mkdir(parents=True, exist_ok=True)
if not any(val.iterdir()):
    # no real captures: 20 CIDIS val frames (never trained on), 4x-downsampled
    val = ROOT / 'val_cidis'; val.mkdir(exist_ok=True)
    if not any(val.iterdir()):
        import cv2
        src = sorted(pathlib.Path('/content/cidis/dataset/thermal/val').glob('*.bmp'))[:20]
        if not src:   # last resort: held-out HR crops
            src = sorted(pathlib.Path('datasets/thermal_hr').glob('*.png'))[-20:]
        if not src:
            raise SystemExit(f'No frames in {ROOT / "val_mag160"}, no CIDIS val and no HR crops — '
                             'put 160×120 captures there, or re-run section 3.')
        for s in src:
            g = cv2.imread(str(s), cv2.IMREAD_GRAYSCALE)
            h, w = g.shape
            cv2.imwrite(str(val / f'{s.stem}.png'),
                        cv2.resize(g, (w // 4, h // 4), interpolation=cv2.INTER_AREA))
        print(f'synthetic val set from {src[0].parent} — real captures in '
              f'{ROOT / "val_mag160"} are better when you have them')


# the newest numbered checkpoint of each stage: stage 1 = faithful, stage 2 = light GAN
CANDIDATES = {}
for tag, stage in (('L1FFT', 'thermal_srvgg_x4_net'), ('lightGAN', 'thermal_srvgg_x4_gan')):
    cks = [c for c in pathlib.Path(f'experiments/{stage}/models').glob('net_g_*.pth') if iter_of(c) >= 0]
    if cks:
        CANDIDATES[tag] = max(cks, key=iter_of)
print({k: v.name for k, v in CANDIDATES.items()})

from IPython.display import Image, display
env = dict(os.environ, PYTHONPATH='/content/Real-ESRGAN')
for tag, ck in CANDIDATES.items():
    out = f'results/val_{tag}_{ck.stem}'
    print(f'\n===== {tag}: {ck}')
    subprocess.run([sys.executable, 'scripts/hallucination_check.py', '--ckpt', str(ck),
                    '--arch', 'srvgg', '--lr', str(val), '--out', out], env=env)
    for p in sorted(pathlib.Path(out).glob('*_check.png'))[:2]:
        display(Image(str(p)))


## 9. Export → ncnn fp16, verify, and pull the models back

Each trained stage at 160×120 and 120×160 (the app's portrait orientation), verified against
PyTorch and zipped as `thermal_ncnn_L1FFT.zip` (plus `thermal_ncnn_lightGAN.zip` if stage 2 ran)
— the packages the app's **Load SR model (.zip)…** takes directly.


In [ ]:
# Restart-safe: rebuilds ROOT / CANDIDATES from disk if an earlier cell's state is gone.
import os, sys, pathlib, subprocess, zipfile
sys.path.insert(0, '/content/Real-ESRGAN/scripts')
from colab_state import restore, iter_of
ROOT, _ = restore(None)
CANDIDATES = {}
for tag, stage in (('L1FFT', 'thermal_srvgg_x4_net'), ('lightGAN', 'thermal_srvgg_x4_gan')):
    cks = [c for c in pathlib.Path(f'experiments/{stage}/models').glob('net_g_*.pth') if iter_of(c) >= 0]
    if cks:
        CANDIDATES[tag] = max(cks, key=iter_of)

env = dict(os.environ, PYTHONPATH='/content/Real-ESRGAN')
PACKAGES = {}
for tag, ck in CANDIDATES.items():
    out = pathlib.Path(f'export/{tag}'); out.mkdir(parents=True, exist_ok=True)
    print(f'\n===== {tag}: {ck}')
    for w, h in ((160, 120), (120, 160)):
        base = f'{out}/thermal_x4_{w}x{h}'
        subprocess.run([sys.executable, 'scripts/export_onnx.py', '--ckpt', str(ck), '--arch', 'srvgg',
                        '--size', f'{w}x{h}', '--out', base], env=env, check=True)
        subprocess.run(['bash', 'scripts/convert_ncnn.sh', base, str(w), str(h), f'thermal_{w}x{h}'],
                       env=env, check=True)
        # PyTorch vs ncnn — fp16 should land under ~6e-3 max, 1e-3 mean (notes in the script)
        subprocess.run([sys.executable, 'scripts/verify_ncnn.py', '--ref', f'{base}_ref.pt',
                        '--param', f'{out}/thermal_{w}x{h}_fp16.param',
                        '--bin', f'{out}/thermal_{w}x{h}_fp16.bin'], env=env)
    z = pathlib.Path(f'/content/thermal_ncnn_{tag}.zip')
    with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(out.glob('thermal_*_fp16.*')):
            zf.write(f, f.name)
    PACKAGES[tag] = (z, ck)
    print('packaged', z)


In [ ]:
# what was packaged
for tag, (z, ck) in PACKAGES.items():
    print(f'{tag:9s} {z.name:28s} {z.stat().st_size / 1e6:.2f} MB   from {ck}')


In [ ]:
# copy the packages and their checkpoints to Drive, and offer direct downloads
import shutil
from google.colab import files
(ROOT / 'export').mkdir(parents=True, exist_ok=True)
for tag, (z, ck) in PACKAGES.items():
    shutil.copy(z, ROOT / 'export' / z.name)
    shutil.copy(ck, ROOT / 'export' / f'{tag}_{ck.name}')
    files.download(str(z))


---
### What to send back

`thermal_ncnn_L1FFT.zip` (and `thermal_ncnn_lightGAN.zip` if stage 2 ran) — load it in the app's
drawer — plus the `results/val_*/summary.csv` numbers. The `.pth` checkpoints on Drive are worth
keeping too: re-converting is free, re-training is not.
